In [1]:
import requests
import pandas as pd

BASE_URL = "http://new-service:8000"


In [2]:
# Fetch all accounts
accounts_response = requests.get(f"{BASE_URL}/accounts/", params={"limit": 1000})
accounts_response.raise_for_status()
accounts = accounts_response.json().get("accounts", [])

print(f"{len(accounts)} cuenta(s) encontrada(s)")


5 cuenta(s) encontrada(s)


In [3]:
# Fetch transactions for every account and keep only income
all_transactions = []

for account in accounts:
    account_id = account["id"]
    txn_response = requests.get(f"{BASE_URL}/accounts/{account_id}/transactions")
    txn_response.raise_for_status()
    transactions = txn_response.json().get("transactions", [])
    all_transactions.extend(transactions)

txn_df = pd.DataFrame(all_transactions)
print(f"{len(txn_df)} transacción(es) obtenida(s) en total")


527 transacción(es) obtenida(s) en total


In [4]:
# Filter income and parse dates
income_df = txn_df[txn_df["transaction_type"] == "income"].copy()

income_df["transaction_date"] = pd.to_datetime(
    income_df["transaction_date"], errors="coerce"
)
income_df["amount"] = pd.to_numeric(income_df["amount"], errors="coerce")
income_df = income_df[income_df["amount"].ne(0)].copy()
income_df = income_df.dropna(subset=["transaction_date", "amount", "account_id"])

income_df = income_df.sort_values(["account_id", "transaction_date"]).reset_index(drop=True)


In [5]:
if income_df.empty:
    print("Sin ingresos registrados.")
else:
    display_df = (
        income_df[["account_id", "transaction_date", "amount", "description"]]
        .copy()
    )
    display_df["transaction_date"] = display_df["transaction_date"].dt.strftime("%d/%m/%Y")

    for account_id, group in display_df.groupby("account_id"):
        print(f"\nCuenta: {account_id}")
        display(
            group.drop(columns="account_id")
            .reset_index(drop=True)
            .style
            .format({"amount": "{:,.2f}"})
            .set_caption(str(account_id))
        )



Cuenta: 193-37565662-0-11


,transaction_date,amount,description
0,02/01/2025,2.50,Pago YAPE de 19191
1,20/01/2025,60.09,TRAN.CTAS.PROP.BM
2,20/01/2025,"12,100.00",TRAN.CTAS.PROP.BM
3,25/01/2025,100.00,Pago YAPE de 19172
4,27/01/2025,100.00,Pago YAPE de 19172
5,04/02/2025,15.00,Pago YAPE de 19390
6,05/02/2025,500.00,Pago YAPE de 19172
7,07/02/2025,500.00,Pago YAPE de 19172
8,21/02/2025,203.00,Pago YAPE de 19300
9,03/03/2025,43.07,TRAN.CTAS.PROP.BM
